# AcuDockMetal — Metal-Aware ε-Certified Molecular Docking

A novel docking system combining:
- **Metal-aware scoring** with coordination chemistry modeling for 9 biologically relevant metals
- **ε-certified optimality** guarantees via branch-and-bound search
- **Multi-fidelity rescoring** (classical → ML → QM/MM)

This notebook clones the AcuDockMetal repository, installs dependencies, and provides an interactive interface to the full pipeline.

## 1. Environment Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/Grimlock5310/AcuDockMetal.git 2>/dev/null || (cd AcuDockMetal && git pull)
%cd AcuDockMetal

# Install Python dependencies
!pip install -q numpy scipy rdkit-pypi biopython meeko vina pdbfixer openmm \
    openbabel-wheel prody py3Dmol matplotlib seaborn pandas

# Install GNINA binary
!wget -q https://github.com/gnina/gnina/releases/latest/download/gnina \
    -O /usr/local/bin/gnina 2>/dev/null && chmod +x /usr/local/bin/gnina

# Optional: xtb for QM/MM rescoring (Level 4)
!pip install -q xtb-python 2>/dev/null || echo 'xtb-python not available; Level 4 QM/MM will be skipped'

print('Setup complete!')

## 2. Import & Configure

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from acudockmetal import (
    MetalParameterLibrary,
    ChelatingGroupDetector,
    MetalSiteDetector,
    CoordinationHypothesisGenerator,
    ReceptorPreparator,
    LigandPreparator,
    VinaEngine,
    GninaEngine,
    DockingOrchestrator,
    MetalAwareScorer,
    CertifiedDockingEngine,
    MultiFidelityRescorer,
    GeometryValidator,
    DockingVisualizer,
    AcuDockMetalPipeline,
)
from acudockmetal.pipeline import AcuDockConfig

print('All modules imported successfully!')

# Show supported metals
lib = MetalParameterLibrary()
print(f'\nSupported metals: {lib.supported_metals}')
print(f'\n{lib.summary()}')

In [ ]:
# Pipeline configuration
config = AcuDockConfig(
    exhaustiveness=32,
    num_poses=20,
    box_padding=10.0,
    epsilon=1.0,
    run_certification=True,
    max_rescore_level=3,        # Set to 4 to enable QM/MM (needs xtb)
    gnina_enabled=True,
    enumerate_tautomers=True,
    ph=7.4,
)

pipeline = AcuDockMetalPipeline(config)
print('Pipeline configured.')

## 3. Quick Test — Chelating Group Detection

Before running a full docking, verify the chelator detector on known metal-binding groups.

In [ ]:
detector = ChelatingGroupDetector(min_confidence=0.3)

# Hydroxamic acid (classic ZBG for MMPs/HDACs)
donors = detector.detect_from_smiles('ONC(=O)c1ccccc1')
print('Hydroxamic acid:')
print(detector.summary(donors))

print()

# Vorinostat (SAHA) - HDAC inhibitor
donors = detector.detect_from_smiles('ONC(=O)CCCCCCC(=O)Nc1ccccc1')
print('Vorinostat (SAHA):')
print(detector.summary(donors))

print()

# Catechol
donors = detector.detect_from_smiles('Oc1ccccc1O')
print('Catechol:')
print(detector.summary(donors))

## 4. Fetch a Metalloprotein Target

Download a zinc-containing structure from the PDB. Default: **1T6E** (MMP-13 with hydroxamic acid inhibitor).

In [ ]:
import os, urllib.request

PDB_ID = '1T6E'  # @param {type:"string"}
pdb_path = f'{PDB_ID}.pdb'

if not os.path.exists(pdb_path):
    url = f'https://files.rcsb.org/download/{PDB_ID}.pdb'
    urllib.request.urlretrieve(url, pdb_path)
    print(f'Downloaded {PDB_ID}.pdb')
else:
    print(f'{PDB_ID}.pdb already exists')

# Define ligand SMILES (hydroxamic acid inhibitor)
LIGAND_SMILES = 'ONC(=O)c1cc(Cc2ccccc2)on1'  # @param {type:"string"}

print(f'\nTarget: {PDB_ID}')
print(f'Ligand SMILES: {LIGAND_SMILES}')

## 5. Run the Full Pipeline

In [ ]:
result = pipeline.run(
    receptor_path=pdb_path,
    ligand_input=LIGAND_SMILES,
    output_dir='./output',
)

print(result.summary)

## 6. Visualize Results

In [ ]:
# 3D view of the metal site
if result.metal_sites:
    view = pipeline.show_metal_site(result)
    if view:
        view.show()
else:
    print('No metal sites detected.')

In [ ]:
# 3D view of best docked pose
if result.docking_poses:
    view = pipeline.show_best_pose(result)
    if view:
        view.show()
else:
    print('No docking poses generated.')

In [ ]:
# 2D analysis plots
plots = pipeline.plot_results(result)

import matplotlib.pyplot as plt
for name, fig in plots.items():
    print(f'\n--- {name} ---')
    plt.figure(fig.number)
    plt.show()

## 7. Detailed Inspection

Explore individual components in more detail.

In [ ]:
# Metal site details
site_detector = MetalSiteDetector()
print(site_detector.summary(result.metal_sites))

# Hypotheses
hyp_gen = CoordinationHypothesisGenerator()
print(f'\n{hyp_gen.summary(result.hypotheses)}')

# Validation
validator = GeometryValidator()
print(f'\n{validator.summary(result.validations)}')

# Certification
if result.certification:
    print(f'\n{result.certification.summary}')

## 8. Custom Ligand

Try your own ligand SMILES against the same target.

In [ ]:
CUSTOM_SMILES = 'ONC(=O)CCCCCCC(=O)Nc1ccccc1'  # @param {type:"string"}

# Detect chelating groups first
donors = detector.detect_from_smiles(CUSTOM_SMILES)
print('Chelating groups detected:')
print(detector.summary(donors))

# Filter for zinc compatibility
zn_donors = detector.compatible_with_metal(donors, 'ZN')
print(f'\nZn-compatible groups: {len(zn_donors)}')
print(f'Total denticity for Zn: {detector.total_denticity(zn_donors)}')

In [ ]:
# Run pipeline with custom ligand
custom_result = pipeline.run(
    receptor_path=pdb_path,
    ligand_input=CUSTOM_SMILES,
    output_dir='./output_custom',
)

print(custom_result.summary)